 # CAPA Dashboard - SQL Queries

Connects to the `medfluss` PostgreSQL database and runs KPI queries for Tableau.

**Input:** `data/capa_medfluss.csv` loaded into PostgreSQL

**Output:** 7 DataFrames: df_status, df_severity, df_department, df_rootcause, df_overdue, df_effectiveness, df_trend


In [1]:
from pathlib import Path
import psycopg2
from sqlalchemy import create_engine
import pandas as pd

In [2]:
# Load Data
DATA_DIR = Path('data')
capa_df = pd.read_csv(DATA_DIR / 'capa_medfluss.csv')

In [3]:
capa_df.head()

,capa_id,company,recall_id,product_res_number,recalling_firm,product_description,nonconformity,original_root_cause,corrected_root_cause,department,assigned_owner,severity,capa_status,open_date,close_date,days_open,overdue,corrective_action,effectiveness_result
0,CAPA-MF-1000,MedFluss GmbH,85112,Z-0147-2010,Hospira Inc,Power cord for QVue Continuous Cardiac Output ...,Fire/Shock hazard-- The power cord used in the...,Component design/selection,Component design/selection,Quality Assurance,Thomas Becker,Major,Closed,2022-08-21,2022-11-22,93,No,Hospira initiated its recall on 08/14/2009. A...,Effective
1,CAPA-MF-1001,MedFluss GmbH,105064,Z-0268-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply, Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Lisa Hartmann,Major,Closed,2022-07-10,2022-10-30,112,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Partially Effective
2,CAPA-MF-1002,MedFluss GmbH,105401,Z-0269-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Carlos Rivera,Major,Closed,2023-09-01,2023-09-22,21,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Effective
3,CAPA-MF-1003,MedFluss GmbH,104072,Z-3284-2011,Hospira Inc.,Plum A+ Single Channel Infusion Pumps; Hospira...,Hospira has received reports of incorrect seat...,Nonconforming Material/Component,Nonconforming Material/Component,Quality Assurance,Thomas Becker,Major,Closed,2025-11-10,2026-01-16,67,No,"Hospira, Inc. sent an ""URGENT DEVICE RECALL"" l...",Effective
4,CAPA-MF-1004,MedFluss GmbH,107986,Z-1338-2012,Medtronic Neuromodulation,"Medtronic, Model 8870, Application Software Ca...",Medtronic has confirmed that an algorithm used...,Software design,Software design,Quality Assurance,Thomas Becker,Critical,Closed,2022-08-28,2022-10-12,45,No,"Medtronic mailed an ""Urgent Medical Device Cor...",Effective


In [4]:
# Database config
DB_NAME = 'medfluss'
DB_USER = 'jetalbhanarkar'
DB_HOST = 'localhost'
DB_PORT = '5432'

In [5]:
# Connect
conn = psycopg2.connect(
    dbname = DB_NAME,
    user = DB_USER,
    host = DB_HOST,
    port = DB_PORT
)

print('Connected to medfluss successfully')

Connected to medfluss successfully


In [6]:
from sqlalchemy import create_engine

In [7]:
engine = create_engine(f'postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [8]:
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS capa_medfluss")
conn.commit()
cursor.close()
print("Table dropped")

Table dropped


In [9]:
# load CAPA table
capa_df.to_sql('capa_medfluss', engine, if_exists = 'replace', index = False)
print('CAPA table loaded successfully')

CAPA table loaded successfully


## KPI Queries

In [10]:
# CAPA Status Breakdown
query = """
SELECT capa_status,
    COUNT(*) AS  total_capas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM capa_medfluss
GROUP BY capa_status
ORDER BY total_capas DESC;
"""

df_status = pd.read_sql(query, engine)
df_status

,capa_status,total_capas,percentage
0,Closed,417,78.83
1,Open,112,21.17


In [11]:
# Severity Breakdown
query = """
SELECT
    severity,
    COUNT(*) AS total_capas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM capa_medfluss
GROUP BY severity
ORDER BY total_capas DESC;
"""

df_severity = pd.read_sql(query, engine)
df_severity

,severity,total_capas,percentage
0,Major,275,51.98
1,Minor,187,35.35
2,Critical,67,12.67


In [12]:
# Department Analysis
query = """
  SELECT
      department,
      COUNT(*) AS total_capas,
      SUM(CASE WHEN capa_status = 'Open' THEN 1 ELSE 0 END) AS open_capas,
      SUM(CASE WHEN capa_status = 'Closed' THEN 1 ELSE 0 END) AS closed_capas,
      SUM(CASE WHEN overdue = 'Yes' THEN 1 ELSE 0 END) AS overdue_capas
  FROM capa_medfluss
  GROUP BY department
  ORDER BY total_capas DESC;
  """

df_department = pd.read_sql(query, engine)
df_department

,department,total_capas,open_capas,closed_capas,overdue_capas
0,Quality Assurance,329,56,273,30
1,Design & Development,102,17,85,9
2,Manufacturing,98,39,59,22


In [13]:
# Root Cause Distribution
query = """
  SELECT
      corrected_root_cause,
      COUNT(*) AS total_capas,
      ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
  FROM capa_medfluss
  GROUP BY corrected_root_cause
  ORDER BY total_capas DESC
  LIMIT 10;
  """

df_rootcause = pd.read_sql(query, engine)
df_rootcause

,corrected_root_cause,total_capas,percentage
0,Device Design,102,19.28
1,Process control,98,18.53
2,Nonconforming Material/Component,88,16.64
3,Under Investigation by firm,66,12.48
4,Component design/selection,48,9.07
5,Software design,30,5.67
6,Employee error,20,3.78
7,Process design,20,3.78
8,Labeling design,15,2.84
9,Software change control,7,1.32


# Overdue CAPAs by Severity and Department
query = """
SELECT
    severity,
    department,
    COUNT(*) AS total_open,
    SUM(CASE WHEN overdue = 'Yes' THEN 1 ELSE 0 END) AS overdue_count,
    ROUND(AVG(days_open)::numeric) AS avg_days_open
FROM capa_medfluss
WHERE capa_status = 'Open'
GROUP BY severity, department
ORDER BY overdue_count DESC;
"""

df_overdue = pd.read_sql(query, engine)
df_overdue

In [ ]:
# Effectiveness of Closed CAPAs
query = """
SELECT
    effectiveness_result,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM capa_medfluss
WHERE capa_status = 'Closed'
GROUP BY effectiveness_result
ORDER BY total DESC;
"""

df_effectiveness = pd.read_sql(query, engine)
df_effectiveness

,effectiveness_result,total,percentage
0,Effective,270,64.75
1,Partially Effective,108,25.90
2,Not Effective,39,9.35


In [ ]:
# CAPA Trend by Year
query = """
SELECT
    TO_CHAR(open_date::date, 'YYYY') AS year,
    COUNT(*) AS capas_opened
FROM capa_medfluss
WHERE open_date IS NOT NULL
GROUP BY year
ORDER BY year;
"""

df_trend = pd.read_sql(query, engine)
df_trend

,year,capas_opened
0,2022,104
1,2023,97
2,2024,108
3,2025,165
4,2026,55
